# Suite DOM — Value domains

A value domain names a standard's representation, with only its parameters. Domains are data: a standard dialect
(ISO/IEC 11404's, IEEE 754's, IEEE 1164's or Unicode's), a singleton, owns each by name, and each one's schema is a named
`Schemas.OfAny` of the scalar it narrows, related to it both ways; they are stored and round-trip, and the framework's
interface reads them (`kindOf`, `properties`, `nameOf`): a domain's kind, by its dialect's tag, and its properties,
its parameters and the ones inferred from them, populated by hand. A language's own kinds are its dialect's too. A term
carries no domain, so evaluation takes values by the schemas they are written with, ints unbounded and floats binary64,
and inference gives the natives' domains, each the values of one native type, by the signatures of Patterns' operations
over them and over collections. IEEE 754's arithmetic, which C's and SystemVerilog's evaluators use, is tested here
directly.

In [ ]:
import { Framework } from "@mbse/expressions";
import { Evaluators as V, Expressions as E } from "@mbse/expressions/Patterns/Expressions";
import * as IEEE1164 from "@mbse/expressions/IEEE1164/Expressions";
import * as IEEE754 from "@mbse/expressions/IEEE754/Expressions";
import * as ISOIEC11404 from "@mbse/expressions/ISOIEC11404/Expressions";
import * as Unicode from "@mbse/expressions/Unicode/Expressions";
import * as CDialect from "@mbse/expressions/C/Expressions";
import * as Cpp from "@mbse/expressions/Cpp/Expressions";
import * as Excel from "@mbse/expressions/Excel/Expressions";
import * as Latex from "@mbse/expressions/LaTeX/Expressions";
import * as Matlab from "@mbse/expressions/MATLAB/Expressions";
import * as Python3 from "@mbse/expressions/Python/Python3";
import * as SystemVerilog from "@mbse/expressions/SystemVerilog/Expressions";
import * as TypeScript5 from "@mbse/expressions/TypeScript/TypeScript5";
import * as Patterns from "@mbse/expressions/Patterns/Expressions";
import * as Translators from "@mbse/expressions/Translators";
import { Domains as FD, Evaluators as FE, Terms as F } from "@mbse/expressions/Framework";
import { Builders, Comparison as C, Errors, JSON, Validators, YAML } from "@mbse/schemas/Framework";
import { assert, equal, map, raises, same } from "./support.js";

const { Domains: D, Ieee754 } = Patterns;
const { LookupError, NotImplementedError, ValueError } = Errors;

const INSTALLS = [Framework.install, ISOIEC11404.install, IEEE754.install, IEEE1164.install, Unicode.install, Patterns.install];
const store: any = new Builders.Store();
for (const install of INSTALLS) store.transaction(install); // the framework, the standard dialects, then Patterns: each its own transaction
store.transaction(Patterns.define); // Patterns' operations, by name, which validation reads

/** The term `spec` writes, the root of an expression built in a transaction. */
function term(spec: unknown): any {
  return store.transaction(() => F.term(store.Expressions.Framework.Expression((e: any) => e.root(E.DIALECT.spec(spec)))));
}

/** The value of the term `spec` writes, with the variables in `scope` bound. */
function val(spec: unknown, scope: Record<string, unknown> = {}): any {
  return V.OfAny(term(spec), scope);
}

/** The value of a Patterns call, with the variables in `scope` bound. */
function evIn(scope: Record<string, unknown>, name: string, ...args: unknown[]): any {
  return val(E.operation(name, ...args), scope);
}

/** The value of a Patterns call. */
function ev(name: string, ...args: unknown[]): any {
  return evIn({}, name, ...args);
}

const bytes = (...values: number[]) => new Uint8Array(values);
const x = E.variable("x");

<a id="DOM-01"></a>
## DOM-01 · Inference's natives' domains are each the values of one native type; a value's is its type's, or Object

In [ ]:
{
  // The natives' domains, the framework's OfTypes, each named by its type, and a native's domain its type's.
  const NATIVES: [FD.Domain, string, unknown][] = [[D.Bool, "bool", true], [D.Int, "int", 1n], [D.Float, "float", 1.5],
    [D.Str, "str", "a"], [D.Bytes, "bytes", bytes(97)]];
  assert(NATIVES.every(([domain, name]) => domain instanceof FD.OfTypes && domain.name() === name && String(domain) === name));
  assert(same(NATIVES.map(([, , value]) => D.of(value)), NATIVES.map(([domain]) => domain)));
  // Each holds the values of exactly its type: a bool is not an int, nor an int a float; ints are unbounded.
  assert(D.Int.contains(10n ** 30n) && !D.Int.contains(true) && !D.Bool.contains(1n) && !D.Float.contains(1n));
  assert(D.Str.contains("日本語 🚀") && !D.Str.contains(bytes(97)) && D.Bytes.contains(bytes()) && !D.Bytes.contains("abc"));
  // A domain includes only itself: Patterns has no implied promotions.
  assert(D.Int.includes(D.Int) && !D.Int.includes(D.Float) && !D.Float.includes(D.Int) && !D.Int.includes(D.Bool));
  assert(D.Float.includes(new FD.OfUnion(D.Float)) && !D.Int.includes(new FD.OfUnion(D.Int, D.Float)) && FD.overlaps(FD.Anything, D.Int));
  // Any other value of the dialect is a data object, a term among them; None is no value of it.
  assert(D.of(term(E.variable("x"))) === D.Object && D.Object.name() === "object");
  raises(TypeError, () => D.of(null), "a NoneType is not a value of the Patterns dialect");
}

<a id="DOM-03"></a>
## DOM-03 · Domains are data: a dialect owns each by name, and each one's schema, an OfAny of its scalar, reads it

In [ ]:
const DIALECTS = new Map<any, string>([[ISOIEC11404, "ISOIEC11404"], [IEEE754, "IEEE754"], [IEEE1164, "IEEE1164"], [Unicode, "Unicode"]]); // each by its family
const STANDARD = new Map<string, Map<string, string>>([ // the standard dialects' domains, each the scalar its schema narrows
  ["ISOIEC11404", new Map([["boolean", "bool"], ["integer", "int"], ["octetstring", "bytes"], ["octet", "int"], ["modulo(256)", "int"]])],
  ["IEEE754", new Map([["binary64", "float"], ["binary32", "float"]])], ["IEEE1164", new Map([["std_logic", "str"]])],
  ["Unicode", new Map([["string", "str"]])]]);
const defined = new Map<string, any>([...DIALECTS].map(([module, family]) => [family, store.transaction(module.define)[0]])); // each dialect's domains
const singletons = new Map<string, any>([...STANDARD.keys()].map((family) => [family, store.Instances.Expressions[family].Expressions])); // by its path
assert([...STANDARD.keys()].every((family) => singletons.get(family) === defined.get(family).get("dialect")));
const iso = singletons.get("ISOIEC11404");
const domains = new Map<string, any>([...STANDARD.keys()].flatMap((family) =>
  [...singletons.get(family).domains].map((e: any): [string, any] => [`${e.name}`, e.domain])));
assert(equal(new Map([...STANDARD.keys()].map((family) => [family, [...singletons.get(family).domains].map((e: any) => `${e.name}`)])),
  new Map([...STANDARD].map(([family, names]) => [family, [...names.keys()]]))));
assert(iso.domains({ name: "octet" }) === domains.get("octet") && domains.get("octet") === defined.get("ISOIEC11404").get("octet"));
assert(store.transaction(ISOIEC11404.define)[0].get("octet") === domains.get("octet")); // again: nothing
assert(equal(new Map([...domains].map(([name, domain]) => [name, FD.kindOf(domain)[1]])), new Map([ // each its kind, read through the interface
  ["boolean", "boolean"], ["integer", "integer"], ["octetstring", "octetstring"], ["octet", "octet"], ["modulo(256)", "modulo"],
  ["binary64", "ieee754"], ["binary32", "ieee754"], ["std_logic", "ieee1164"], ["string", "unicode"]])));
const scalarSchemas: Record<string, string> = { bool: "Schemas.OfBool", int: "Schemas.OfInt", float: "Schemas.OfFloat", str: "Schemas.OfStr", bytes: "Schemas.OfBytes" };
const SPELLED = new Map([["modulo(256)", "modulo_256"]]); // a schema's name is a dotted identifier: 11404's notation spelled as one
for (const [family, names] of STANDARD) { // each one's schema: an OfAny of its scalar, its domain read both ways
  for (const [name, scalar] of names) {
    const schema = store.registered(`Expressions.${family}.Expressions.${SPELLED.get(name) ?? name}`);
    assert(same(schema.schemas, [store.registered(scalarSchemas[scalar] as string)]) && schema.validate().length === 0);
    assert(same([...domains.get(name).schema].map((e: any) => e.schema), [schema]));
    const read = [...new Builders.OfAny.Builder(schema, false, store).domain].map((e: any) => e.domain);
    assert(equal(read.map((d: any) => [FD.nameOf(d), FD.kindOf(d).slice(0, 2), FD.properties(d)]), [
      [name, FD.kindOf(domains.get(name)).slice(0, 2), FD.properties(domains.get(name))]]));
  }
}
const [Domain, HasDomain]: any[] = [store.registered("Expressions.Framework.Domain"), store.registered("Expressions.Framework.HasDomain")];
assert(equal(HasDomain.uniques.map((u: any) => [...u]), [["schema"], ["domain"]]) && Domain.adjacencies.get("dialect").owner);
// The framework's union of kinds is of the dialects' unions, each named by its dialect's tag.
const DomainKind = store.registered("Expressions.Framework.DomainKind");
assert(same(DomainKind.branches.map((b: any) => b.name), ["isoiec11404", "ieee754", "ieee1164", "unicode"]));
assert(same(DomainKind.branches[0].schema.branches.map((b: any) => b.name), [
  "boolean", "integer", "modulo", "bitstring", "octet", "octetstring", "state", "packed"]));
// A domain of its own, owned by its dialect by name; a packed one links its enum and its representation, domains too.
const [state, coded, bad, empty] = store.transaction(() => {
  const state = store.Expressions.Framework.Domain((d: any) => d.kind((k: any) => k.as_value((a: any) => a.as_union(
    (u: any) => u.isoiec11404((v: any) => v.state((e: any) => e.members(["IDLE", "RUN"]).ordered(false)))))));
  const coded = store.Expressions.Framework.Domain((d: any) => d.kind((k: any) => k.as_value((a: any) => a.as_union((u: any) => u.isoiec11404(
    (v: any) => v.packed((e: any) => e.codes([0n, 1n]).ordered(true)))))).packs((a: any) => a.insert(
    (e: any) => e.domain(state).representation(domains.get("octet")))));
  const bad = store.Expressions.Framework.Domain((d: any) => d.kind((k: any) => k.as_value((a: any) => a.as_union(
    (u: any) => u.isoiec11404((v: any) => v.integer((e: any) => e.width(0n)))))));
  const empty = store.Expressions.Framework.Domain((d: any) => d);
  store.Expressions.ISOIEC11404.Expressions.Dialect(iso, (b: any) => b.domains({
    state: (e: any) => e.domain(state), coded: (e: any) => e.domain(coded), bad: (e: any) => e.domain(bad),
    empty: (e: any) => e.domain(empty) })); // each owned: written with its owner
  return [state, coded, bad, empty];
});
const OWN = [...(STANDARD.get("ISOIEC11404") as Map<string, string>).keys(), "state", "coded", "bad", "empty"];
assert(same([...iso.domains].map((e: any) => `${e.name}`), OWN));
assert(same([state, coded, bad].map((domain) => FD.kindOf(domain)[1]), ["state", "packed", "integer"]));
assert(equal([...coded.packs].map((e: any) => [e.domain, e.representation]), [[state, domains.get("octet")]])); // packed into octet codes
// A width is data: that it is positive is a constraint for a later resolver, not the store's.
assert(equal(FD.properties(bad), new Map([["width", 0n]])) && FD.nameOf(bad) === "bad");
raises(ValueError, () => FD.kindOf(empty), "a domain needs a kind");

<a id="DOM-05"></a>
## DOM-05 · Over the wire a domain is its dialect's, by name: it round-trips, and so does a store holding it

In [ ]:
{
  const Dialect = store.registered("Expressions.ISOIEC11404.Expressions.Dialect");
  // Each a document of its own, by its path: read back, it is the object the store holds. Its kind is written its
  // dialect's, then its kind's.
  assert(JSON.ToJSON(store)(Domain, domains.get("octet")).includes('"isoiec11404": {"octet": {"ordered": true}}'));
  assert(JSON.ToJSON(store)(Domain, domains.get("modulo(256)")).includes(
    '"isoiec11404": {"modulo": {"modulus": 256, "least": 0, "greatest": 255, "ordered": true}}'));
  assert(JSON.ToJSON(store)(Domain, domains.get("binary32")).includes('"ieee754": {"ieee754": {"format": "binary32", "rounding": "roundTiesToEven",'));
  for (const domain of [domains.get("octet"), domains.get("modulo(256)"), domains.get("binary32"), state, coded]) {
    for (const [encode, decode] of [[JSON.ToJSON(store), JSON.FromJSON(store)], [YAML.ToYAML(store), YAML.FromYAML(store)]] as [any, any][]) {
      const back = decode(Domain, encode(Domain, domain));
      assert(back === domain && equal(FD.properties(back), FD.properties(domain))); // owned: its path, its dialect's and its name, finds it
    }
  }
  const back: any = JSON.FromJSON(store)(Dialect, JSON.ToJSON(store)(Dialect, iso));
  assert(back === iso && same([...back.domains].map((e: any) => `${e.name}`), OWN));
  // A store holding them round-trips whole, after its documents were read back, into a store with the same installs: nothing lost, nothing written twice.
  const snapshot = JSON.ToJSON(store).Snapshot();
  const again: any = new Builders.Store();
  for (const install of INSTALLS) again.transaction(install);
  JSON.FromJSON(again).Snapshot(snapshot);
  assert(JSON.ToJSON(again).Snapshot() === snapshot);
  const restored = new Map<string, any>([...again.Instances.Expressions.ISOIEC11404.Expressions.domains].map((e: any) => [`${e.name}`, e.domain])); // keyed by name
  assert(same([...restored.keys()].sort(), [...OWN].sort()));
  assert(equal(["state", "coded", "bad"].map((name) => [FD.kindOf(restored.get(name))[1], FD.properties(restored.get(name))]), [
    ["state", new Map<string, unknown>([["members", ["IDLE", "RUN"]], ["ordered", false]])],
    ["packed", new Map<string, unknown>([["codes", [0n, 1n]], ["ordered", true]])], ["integer", new Map([["width", 0n]])]]));
  assert(equal([...restored.get("coded").packs].map((e: any) => [e.domain, e.representation]), [[restored.get("state"), restored.get("octet")]]));
  // A term carries no domain: a literal holds a value, its spelling and its problems, and its value's schema is its own.
  assert(same([...store.registered("Expressions.Framework.Literal").properties.keys()], ["value", "text", "problems"])
    && E.literal.length === 1);
}

<a id="DOM-06"></a>
## DOM-06 · Evaluation compares values by their schemas: of one scalar, else incomparable, and floats as mbse-schemas does

In [ ]:
{
  // Evaluation compares values by the schemas they are written with: of one scalar, else incomparable.
  assert(ev("lt", -1n, 1n) === true && ev("eq", 5n, 5.0) === null && ev("eq", "RUN", "RUN") === true);
  assert(ev("lt", bytes(0, 1), bytes(1, 0)) === true); // bytes by their octets
  // Floats compare as mbse-schemas compares floats: NaNs equal each other, and -0 comes before 0.
  assert(ev("eq", NaN, NaN) === true && ev("lt", NaN, 1.0) === null && ev("lt", -0, 0) === true);
}

<a id="DOM-07"></a>
## DOM-07 · Evaluation's ints are unbounded and its floats binary64, and it does not promote

In [ ]:
{
  // Evaluation takes ints by their schema, unbounded, and floats as binary64; it does not promote.
  assert(ev("add", 2n ** 70n, 1n) === 2n ** 70n + 1n && ev("neg", -128n) === 128n && ev("mul", 1.5, 2.0) === 3.0);
  raises(TypeError, () => ev("add", 1n, 1.5), "add expects numbers of one domain, got int and float");
  raises(TypeError, () => ev("neg", "a"), "neg expects a number, got str");
}

<a id="DOM-08"></a>
## DOM-08 · The bitwise operations take ints, as two's complement of infinite width, and shifts an int count

In [ ]:
{
  assert(ev("bitand", 12n, 10n) === 8n && ev("bitor", 12n, 10n) === 14n && ev("bitxor", 12n, 10n) === 6n && ev("bitnot", 5n) === -6n);
  assert(ev("bitand", -1n, 15n) === 15n && ev("bitnot", -1n) === 0n && ev("bitor", -16n, 15n) === -1n);
  assert(ev("shl", 3n, 4n) === 48n && ev("shr", -7n, 1n) === -4n && ev("shl", 1n, 100n) === 2n ** 100n); // shr toward negative infinity
  raises(TypeError, () => ev("bitand", 1n, bytes(1)), "bitand expects integers of one domain, got int and bytes");
  raises(TypeError, () => ev("bitnot", true), "bitnot expects an integer, got bool");
  raises(ValueError, () => ev("shl", 1n, -1n), "shl needs a non-negative count, got -1");
  raises(TypeError, () => ev("shr", 1n, 1.0), "shr expects an integer count, got float");
  raises(TypeError, () => ev("shl", "a", 1n), "shl expects an integer, got str");
  assert(evIn({ x: null }, "bitand", x, 1n) === null && evIn({ x: null }, "shl", x, 1n) === null);
  assert(evIn({ x: null }, "shl", 1n, x) === null);
}

<a id="DOM-09"></a>
## DOM-09 · Signatures take the natives' domains and collections, so inference follows the domains an environment gives

In [ ]:
function infer(spec: unknown, environment: Record<string, any> = {}): any {
  return E.DIALECT.infer(term(spec), environment);
}

{
  const y = E.variable("y");
  assert(infer(x.eq(y), { x: D.Str, y: D.Str }) === D.Bool && infer(x.lt(y), { x: D.Int, y: D.Int }) === D.Bool);
  raises(TypeError, () => infer(x.lt(y), { x: D.Bool, y: D.Bool }),
    "lt cannot take (bool, bool); it takes (T, T) -> bool for T in int | float | str | bytes");
  assert(infer(x.mul(y), { x: D.Float, y: FD.Anything }) === D.Float); // an unknown takes the other's domain
  raises(TypeError, () => infer(x.add(y), { x: D.Str, y: D.Str }), "add cannot take (str, str); it takes (T, T) -> T for T in int | float");
  raises(TypeError, () => infer(E.operation("neg", "a")), "neg cannot take (str); it takes (T) -> T for T in int | float");
  // The bitwise operations and shifts take ints.
  assert(infer(x.bitxor(y), { x: D.Int, y: D.Int }) === D.Int && infer(E.operation("bitnot", 1n)) === D.Int);
  assert(infer(E.operation("shr", x, 1n), { x: FD.Anything }) === D.Int);
  assert(infer(E.operation("shl", 1n, x), { x: new FD.OfUnion(D.Int, D.Float) }) === D.Int);
  raises(TypeError, () => infer(E.operation("shl", 1n, 1.5)), "shl cannot take (int, float); it takes (int, int) -> int");
  raises(TypeError, () => infer(E.operation("shr", 1.5, 1n)), "shr cannot take (float, int)");
  assert(E.DIALECT.validate(term(E.operation("bitand", 1n, 2n)), [], true).length === 0);
  // A collection's operations take its items' domain: sum numbers, min and max ordered values.
  assert(infer(E.operation("sum", E.list_(1n, 2n))) === D.Int && infer(E.operation("max", E.list_("a", "b"))) === D.Str);
  raises(TypeError, () => infer(E.operation("sum", E.list_("a", "b"))), "sum cannot take (list(str)); it takes (collection of number) -> number");
  raises(TypeError, () => infer(E.operation("min", E.list_(true))),
    "min cannot take (list(bool)); it takes (collection of ordered value) -> ordered value");
}

<a id="DOM-10"></a>
## DOM-10 · An IEEE 754 value is a float up to binary64 and canonical text otherwise, which the format must hold

In [ ]:
{
  const holds: [string, unknown[], unknown[]][] = [ // a format, the values it holds, and values it does not
    ["binary16", [1.5, 65504.0, 2 ** -24, -0, Infinity, NaN], [0.1, 1e10, 2 ** -25, "1.5"]],
    ["binary32", [0.10000000149011612], [0.1]],
    ["binary64", [0.1, 5e-324], ["0.1"]],
    ["binary128", ["0.1", "1E+2", "-0", "Infinity", "-Infinity", "NaN", "1.189731495357231765085759326628007E+4932"],
      ["0.10", "100", "1e2", "-NaN", "1E+5000", "0.1 "]],
    ["decimal64", ["1.50", "1.000000000000000E+16", "0E-398", "-0", "0.000001", "1E-7", "NaN"],
      ["1.5E+0", "12345678901234567", "1E+385", "1E-399", "0.0000001", "+1", "abc"]],
    ["decimal128", ["1.000000000000000000000000000000000", "2E+6111"], ["1E+6112"]],
  ];
  for (const [format, inside, outside] of holds) {
    assert(inside.every((v) => Ieee754.contains(format, v)) && !outside.some((v) => Ieee754.contains(format, v)), format);
  }
  // A decimal value's text is General Decimal Arithmetic's scientific form, keeping the coefficient and the exponent.
  const data: [boolean, bigint, number][] = [[false, 15n, -1], [true, 150n, -2], [false, 1n, 2], [false, 123n, -9], [false, 0n, 3], [true, 0n, 0]];
  assert(same(data.map(([n, c, q]) => Ieee754.text(new Ieee754.Datum(null, n, c, q))), ["1.5", "-1.50", "1E+2", "1.23E-7", "0E+3", "-0"]));
  assert(Ieee754.text(Ieee754.NAN) === "NaN" && same([...Ieee754.FORMATS.get("decimal64")!], [10n, 16, 384]));
  // A binary format's interchange encoding, big-endian, as an unsigned int: the canonical quiet NaN, subnormals, binary128.
  assert(Ieee754.to_bits("binary64", 1.5) === 4609434218613702656n && Ieee754.from_bits("binary64", 4609434218613702656n) === 1.5);
  assert(Ieee754.to_bits("binary64", -0) === 1n << 63n && Ieee754.to_bits("binary32", NaN) === 0x7FC00000n);
  assert(Ieee754.from_bits("binary32", 0x7F800000n) === Infinity && Number.isNaN(Ieee754.from_bits("binary32", 0xFF800001n)));
  assert(Ieee754.from_bits("binary16", 1n) === 2 ** -24 && Ieee754.to_bits("binary16", 65504.0) === 0x7BFFn);
  assert(Ieee754.to_bits("binary16", 2 ** -24) === 1n && Ieee754.to_bits("binary16", 0.0) === 0n);
  assert(Ieee754.to_bits("binary16", -Infinity) === 0xFC00n && Ieee754.WIDTHS.get("binary16") === 16);
  assert(Ieee754.to_bits("binary128", "1") === BigInt("0x3fff" + "00".repeat(14)));
  assert(Ieee754.from_bits("binary128", BigInt("0x3fff8" + "0".repeat(27))) === "1.5");
}

<a id="DOM-11"></a>
## DOM-11 · IEEE 754 arithmetic is exact, then rounded to the format by the rounding direction given

In [ ]:
{
  const ROUNDINGS = ["roundTiesToEven", "roundTiesToAway", "roundTowardPositive", "roundTowardNegative", "roundTowardZero"] as const;
  const [EVEN, AWAY, UP, DOWN, ZERO] = ROUNDINGS;
  const TABLE: [string, string, string, unknown, unknown, unknown][] = [ // format, rounding, operation, operands, result
    ["binary16", EVEN, "add", 1.0, 0.00048828125, 1.0],
    ["binary16", EVEN, "add", 1.0009765625, 0.00048828125, 1.001953125],
    ["binary16", AWAY, "add", 1.0, 0.00048828125, 1.0009765625],
    ["binary16", UP, "add", 1.0, 5.960464477539063e-08, 1.0009765625],
    ["binary16", DOWN, "add", 1.0, 5.960464477539063e-08, 1.0],
    ["binary16", ZERO, "add", 1.0, 5.960464477539063e-08, 1.0],
    ["binary16", DOWN, "add", -1.0, -5.960464477539063e-08, -1.0009765625],
    ["binary16", UP, "add", -1.0, -5.960464477539063e-08, -1.0],
    ["binary16", EVEN, "add", 65504.0, 65504.0, Infinity],
    ["binary16", ZERO, "add", 65504.0, 65504.0, 65504.0],
    ["binary16", UP, "add", 65504.0, 65504.0, Infinity],
    ["binary16", DOWN, "add", 65504.0, 65504.0, 65504.0],
    ["binary16", DOWN, "add", -65504.0, -65504.0, -Infinity],
    ["binary16", UP, "add", -65504.0, -65504.0, -65504.0],
    ["binary16", AWAY, "mul", 65504.0, 2.0, Infinity],
    ["binary16", EVEN, "mul", 6.103515625e-05, 0.0009765625, 5.960464477539063e-08],
    ["binary16", EVEN, "mul", 5.960464477539063e-08, 0.5, 0.0],
    ["binary16", AWAY, "mul", 5.960464477539063e-08, 0.5, 5.960464477539063e-08],
    ["binary16", EVEN, "add", 2047.0, 0.5, 2048.0],
    ["binary16", EVEN, "sub", 1.0, 1.0, 0.0],
    ["binary16", DOWN, "sub", 1.0, 1.0, -0],
    ["binary16", EVEN, "add", -0, -0, -0],
    ["binary16", EVEN, "add", 0.0, -0, 0.0],
    ["binary16", EVEN, "mul", -0, 5.0, -0],
    ["binary16", EVEN, "sub", Infinity, Infinity, NaN],
    ["binary16", EVEN, "add", Infinity, 1.0, Infinity],
    ["binary16", EVEN, "add", 1.0, -Infinity, -Infinity],
    ["binary16", EVEN, "add", NaN, 1.0, NaN],
    ["binary16", EVEN, "mul", Infinity, 0.0, NaN],
    ["binary16", EVEN, "mul", 0.0, -Infinity, NaN],
    ["binary16", EVEN, "mul", Infinity, -2.0, -Infinity],
    ["binary16", EVEN, "mul", NaN, 1.0, NaN],
    ["binary16", EVEN, "neg", 1.5, null, -1.5],
    ["binary16", EVEN, "neg", NaN, null, NaN],
    ["binary16", EVEN, "neg", -0, null, 0.0],
    ["binary32", EVEN, "add", 16777216.0, 1.0, 16777216.0],
    ["binary32", UP, "add", 16777216.0, 1.0, 16777218.0],
    ["binary32", EVEN, "mul", 0.10000000149011612, 3.0, 0.30000001192092896],
    ["binary64", ZERO, "add", 0.1, 0.2, 0.3],
    ["binary64", UP, "add", 0.1, 0.2, 0.30000000000000004],
    ["binary64", DOWN, "mul", 0.3333333333333333, 3.0, 0.9999999999999999],
    ["binary128", EVEN, "add", "0.1", "0.2", "0.30000000000000000000000000000000004"],
    ["binary128", EVEN, "mul", "1E+4932", "1E+1", "Infinity"],
    ["binary128", ZERO, "mul", "1E+4932", "1E+1", "1.189731495357231765085759326628007E+4932"],
    ["binary128", EVEN, "sub", "1", "1", "0"],
    ["binary128", DOWN, "sub", "1", "1", "-0"],
    ["binary128", EVEN, "neg", "-0", null, "0"],
    ["binary128", EVEN, "add", "Infinity", "1", "Infinity"],
    ["binary128", EVEN, "mul", "1E-4960", "1E-10", "0"],
    ["decimal64", EVEN, "add", "1.5", "1.50", "3.00"],
    ["decimal64", EVEN, "mul", "1.5", "1.50", "2.250"],
    ["decimal64", EVEN, "add", "9999999999999999", "1", "1.000000000000000E+16"],
    ["decimal64", EVEN, "add", "9999999999999999", "0.5", "1.000000000000000E+16"],
    ["decimal64", AWAY, "add", "1000000000000000", "0.5", "1000000000000001"],
    ["decimal64", EVEN, "add", "1000000000000000", "0.5", "1000000000000000"],
    ["decimal64", UP, "add", "1000000000000000", "0.1", "1000000000000001"],
    ["decimal64", DOWN, "add", "-1000000000000000", "-0.1", "-1000000000000001"],
    ["decimal64", ZERO, "add", "-1000000000000000", "-0.9", "-1000000000000000"],
    ["decimal64", EVEN, "mul", "1E+369", "1E+10", "1.0000000000E+379"],
    ["decimal64", EVEN, "add", "9.999999999999999E+384", "1E+369", "Infinity"],
    ["decimal64", ZERO, "add", "9.999999999999999E+384", "1E+369", "9.999999999999999E+384"],
    ["decimal64", EVEN, "mul", "1E-398", "0.1", "0E-398"],
    ["decimal64", UP, "mul", "1E-398", "0.1", "1E-398"],
    ["decimal64", EVEN, "mul", "1E-390", "0.00001", "1E-395"],
    ["decimal64", EVEN, "sub", "1.5", "1.5", "0.0"],
    ["decimal64", DOWN, "sub", "1.5", "1.5", "-0.0"],
    ["decimal64", EVEN, "add", "-0", "-0", "-0"],
    ["decimal64", EVEN, "mul", "-0.0", "1E+5", "-0E+4"],
    ["decimal64", EVEN, "mul", "1E+369", "0E+10", "0E+369"],
    ["decimal64", EVEN, "add", "NaN", "1", "NaN"],
    ["decimal64", EVEN, "mul", "Infinity", "0", "NaN"],
    ["decimal64", EVEN, "neg", "1.50", null, "-1.50"],
    ["decimal64", EVEN, "add", "0.000001", "1E-7", "0.0000011"],
    ["decimal64", EVEN, "mul", "123", "1E+5", "1.23E+7"],
    ["decimal128", EVEN, "mul", "0.1", "3", "0.3"],
    ["decimal128", EVEN, "add", "1E+6111", "1E+6111", "2E+6111"],
    ["decimal128", EVEN, "add", "1", "1E-40", "1.000000000000000000000000000000000"],
  ];
  /** The same value: floats by sign too, NaNs alike. */
  const alike = (x: unknown, y: unknown) => Object.is(x, y);
  // Python checks this table against its decimal module and the host's floats too (Python only).
  for (const [format, rounding, operation, a, b, expected] of TABLE) {
    const result = Ieee754.operate(operation, format, rounding, [a, b].filter((v) => v !== null));
    assert(alike(result, expected), `${format} ${a} ${b} ${result}`);
  }
  // Division, which Patterns has no operation for and C has (see C), by the same engine; decimals as divide does.
  const DIVISIONS: [string, unknown, unknown, unknown][] = [ // format, dividend, divisor, quotient, rounding ties to even
    ["binary64", NaN, 1.0, NaN], ["binary64", Infinity, Infinity, NaN], ["binary64", Infinity, -2.0, -Infinity], ["binary64", 1.0, Infinity, 0.0],
    ["binary64", -1.0, Infinity, -0], ["binary64", 1.0, 0.0, Infinity], ["binary64", -1.0, 0.0, -Infinity], ["binary64", 0.0, 0.0, NaN],
    ["binary64", 1.0, 3.0, 0.3333333333333333], ["binary32", 1.0, 3.0, 0.3333333432674408], ["decimal64", "1", "4", "0.25"],
    ["decimal64", "100", "10", "10"], ["decimal64", "1E+2", "1", "1E+2"], ["decimal64", "1", "3", "0.3333333333333333"],
    ["decimal64", "0", "5", "0"], ["decimal64", "0.00", "5", "0.00"], ["decimal64", "1", "Infinity", "0E-398"],
    ["decimal64", "Infinity", "2", "Infinity"], ["decimal64", "1", "0", "Infinity"], ["decimal64", "0", "0", "NaN"],
    ["decimal64", "-1", "8", "-0.125"], ["decimal64", "2.0", "4", "0.5"],
    ["decimal128", "1", "7", "0.1428571428571428571428571428571429"],
  ];
  for (const [format, a, b, expected] of DIVISIONS) assert(alike(Ieee754.operate("div", format, EVEN, [a, b]), expected), `${format} ${a} ${b}`);
  // Evaluation's floats keep the host's arithmetic, which binary64 rounding ties to even is.
  assert(ev("add", 0.1, 0.2) === 0.30000000000000004 && Ieee754.operate("add", "binary64", EVEN, [0.1, 0.2]) === 0.30000000000000004);
  // Conversions between formats and from and to integers, by the same engine (C's and SystemVerilog's casts use them).
  assert(Ieee754.convert("binary64", 0.1, "binary32", EVEN) === 0.10000000149011612);
  assert(Ieee754.convert("binary64", 0.1, "decimal64", EVEN) === "0.1000000000000000");
  assert(Ieee754.convert("binary64", 100.0, "decimal64", EVEN) === "100" && Ieee754.convert("binary64", 0.5, "decimal64", EVEN) === "0.5");
  assert(Ieee754.convert("decimal64", "1.50", "decimal128", EVEN) === "1.50" && Ieee754.convert("decimal64", "0.1", "binary64", EVEN) === 0.1);
  assert(Ieee754.convert("binary64", NaN, "binary128", EVEN) === "NaN" && Ieee754.convert("binary64", -Infinity, "decimal64", EVEN) === "-Infinity");
  assert(Ieee754.from_integer(2n ** 24n + 1n, "binary32", EVEN) === 16777216.0 && Ieee754.from_integer(2n ** 24n + 1n, "binary32", UP) === 16777218.0);
  assert(Ieee754.from_integer(10n ** 20n, "decimal64", EVEN) === "1.000000000000000E+20" && Ieee754.from_integer(-7n, "decimal64", EVEN) === "-7");
  assert(Ieee754.to_integer("binary64", 2.5, EVEN) === 2n && Ieee754.to_integer("binary64", -2.7, ZERO) === -2n);
  assert(Ieee754.to_integer("decimal64", "2.5", EVEN) === 2n && Ieee754.to_integer("binary64", Infinity, EVEN) === Infinity);
  assert(Ieee754.to_integer("binary64", -Infinity, EVEN) === -Infinity);
  raises(ValueError, () => Ieee754.to_integer("binary64", NaN, EVEN), "NaN has no integer value");
}

<a id="DOM-12"></a>
## DOM-12 · IEEE 754 values compare by totalOrder, cohorts by exponent, and NaNs equal each other

In [ ]:
{
  const COMPARISONS: [string, unknown, unknown, number | null][] = [ // a format, two of its values, and how they compare: -1, 0 or 1, or null when unordered
    ["decimal64", "1.5", "1.5", 0], ["decimal64", "1.5", "1.50", 1], ["decimal64", "1.50", "1.5", -1], // one value, two cohorts
    ["decimal64", "-1.5", "-1.50", -1], ["decimal64", "1.5", "2", -1], ["decimal64", "-1", "-2", 1], ["decimal64", "-0", "0", -1],
    ["decimal64", "-Infinity", "-9.999999999999999E+384", -1], ["decimal64", "9.999999999999999E+384", "Infinity", -1],
    ["decimal64", "Infinity", "Infinity", 0], ["decimal64", "NaN", "NaN", 0], ["decimal64", "NaN", "1", null],
    ["decimal64", "1", "NaN", null], ["decimal64", "1", "-1", 1], ["binary128", "0.1", "0.2", -1], ["binary128", "0.1", "0.1", 0],
    ["binary16", -0, 0, -1], ["binary16", NaN, NaN, 0],
  ];
  assert(same(COMPARISONS.map(([format, a, b]) => Ieee754.compare(format, a, b)), COMPARISONS.map(([, , , expected]) => expected)));
  // A binary128 value's text is the shortest that rounds back to it.
  const EVEN = "roundTiesToEven";
  assert(Ieee754.operate("mul", "binary128", EVEN, ["3", "0.5"]) === "1.5");
  assert(Ieee754.operate("add", "binary128", EVEN, ["0.1", "0.2"]) === "0.30000000000000000000000000000000004");
  assert(Ieee754.operate("mul", "binary128", EVEN, ["0.5", "0.5"]) === "0.25");
  assert(Ieee754.operate("neg", "binary128", EVEN, ["0.49"]) === "-0.49");
  assert(Ieee754.operate("mul", "binary128", EVEN, ["1E+48", "1E+1"]) === "1E+49"); // the nearest is below 1E+49
}

<a id="DOM-17"></a>
## DOM-17 · The standard dialects own the kinds and domains, with their inferred properties, read through the framework's interface

In [ ]:
{
  // Each standard dialect: a singleton of its own schema, a branch of the framework's kind union named by its tag, with
  // no terms; its standard by designation; its union of kinds the branch of the framework's named by its tag.
  const KINDS = new Map<any, [string, string, string[]]>([
    [ISOIEC11404, ["isoiec11404", "ISO/IEC 11404:2007", [
      "boolean", "integer", "modulo", "bitstring", "octet", "octetstring", "state", "packed"]]],
    [IEEE754, ["ieee754", "IEEE Std 754-2019", ["ieee754"]]], [IEEE1164, ["ieee1164", "IEEE Std 1164-1993", ["ieee1164"]]],
    [Unicode, ["unicode", "The Unicode Standard", ["unicode"]]]]);
  const Dialects = store.registered("Expressions.Framework.Dialect");
  for (const [module, [tag, standard, kinds]] of KINDS) {
    const path = `Expressions.${DIALECTS.get(module)}.Expressions`;
    assert(same(module.STANDARDS, [standard]) && await import(`@mbse/expressions/${DIALECTS.get(module)}/Expressions`) === module
      && module.Expressions.install === module.install); // its package, by its path, exporting its module's
    const union = new Map<string, any>(DomainKind.branches.map((b: any) => [b.name, b.schema])).get(tag);
    assert(union === store.registered(`${path}.DomainKind`) && same(union.branches.map((b: any) => b.name), kinds));
    const dialect = new Map<string, any>(Dialects.branches.map((b: any) => [b.name, b.schema])).get(tag);
    assert(dialect === store.registered(`${path}.Dialect`) && dialect.ref && dialect.singleton === path);
    assert(same([...dialect.adjacencies.keys()], ["domains", "operations"])
      && FD.operations(singletons.get(DIALECTS.get(module) as string)).size === 0);
    raises(LookupError, () => store.registered(`${path}.Term`), `no schema registered as '${path}.Term'`);
  }
  // The kinds: their parameters, then the properties inferred from them, all optional.
  assert(same([...store.registered("Expressions.ISOIEC11404.Expressions.OfInteger").properties.keys()], [
    "width", "signed", "overflow", "least", "greatest", "ordered"]));
  assert(same([...store.registered("Expressions.IEEE754.Expressions.OfIeee754").properties.keys()], [
    "format", "rounding", "radix", "precision", "emax", "ordered"]));
  assert(same([...store.registered("Expressions.ISOIEC11404.Expressions.OfModulo").properties.keys()], [
    "modulus", "least", "greatest", "ordered"]));
  assert(equal([
    "Expressions.ISOIEC11404.Expressions.OfOctet",
    "Expressions.ISOIEC11404.Expressions.OfBoolean", "Expressions.ISOIEC11404.Expressions.OfBitstring",
    "Expressions.ISOIEC11404.Expressions.OfOctetstring", "Expressions.ISOIEC11404.Expressions.OfState",
    "Expressions.ISOIEC11404.Expressions.OfPacked", "Expressions.IEEE1164.Expressions.OfIeee1164",
    "Expressions.Unicode.Expressions.OfUnicode"].map((name) => [...store.registered(name).properties.keys()]), [
    ["ordered"], ["ordered"], ["width", "ordered"], ["width", "ordered"], ["members", "ordered"], ["codes", "ordered"],
    ["width", "ordered"], ["ordered"]]));
  // Each domain's kind, its dialect's tag then its own, through its dialect's union, and its properties, parameters and
  // inferred ones alike, by name.
  assert(equal(new Map([...domains].map(([name, domain]) => [name, FD.kindOf(domain).slice(0, 2)])), new Map([
    ["boolean", ["isoiec11404", "boolean"]], ["integer", ["isoiec11404", "integer"]],
    ["octetstring", ["isoiec11404", "octetstring"]], ["octet", ["isoiec11404", "octet"]],
    ["modulo(256)", ["isoiec11404", "modulo"]], ["binary64", ["ieee754", "ieee754"]],
    ["binary32", ["ieee754", "ieee754"]], ["std_logic", ["ieee1164", "ieee1164"]], ["string", ["unicode", "unicode"]]])));
  assert(FD.kindOf(domains.get("octet"))[2].as_schema() === store.registered("Expressions.ISOIEC11404.Expressions.OfOctet"));
  assert(FD.kindOf(domains.get("modulo(256)"))[2].as_schema() === store.registered("Expressions.ISOIEC11404.Expressions.OfModulo"));
  const PROPERTIES = new Map<string, Map<string, unknown>>([
    ["boolean", new Map([["ordered", false]])], ["integer", new Map<string, unknown>([["signed", true], ["overflow", "raise"], ["ordered", true]])], // unbounded
    ["octetstring", new Map([["ordered", true]])],
    ["octet", new Map([["ordered", true]])],
    ["modulo(256)", new Map<string, unknown>([["modulus", 256n], ["least", 0n], ["greatest", 255n], ["ordered", true]])],
    ["binary64", new Map<string, unknown>([["format", "binary64"], ["rounding", "roundTiesToEven"], ["radix", 2n], ["precision", 53n], ["emax", 1023n], ["ordered", true]])],
    ["binary32", new Map<string, unknown>([["format", "binary32"], ["rounding", "roundTiesToEven"], ["radix", 2n], ["precision", 24n], ["emax", 127n], ["ordered", true]])],
    ["std_logic", new Map([["ordered", false]])], ["string", new Map([["ordered", true]])]]);
  assert(equal(new Map([...domains].map(([name, domain]) => [name, FD.properties(domain)])), PROPERTIES));
  assert(equal(FD.properties(state), new Map<string, unknown>([["members", ["IDLE", "RUN"]], ["ordered", false]]))); // a container as an array
  assert(equal(FD.properties(coded), new Map<string, unknown>([["codes", [0n, 1n]], ["ordered", true]])));
  // The inferred properties, populated by hand: IEEE 754's agree with its formats' parameters.
  assert(["binary64", "binary32"].map((name) => PROPERTIES.get(name) as Map<string, unknown>).every((p) =>
    equal([p.get("radix"), p.get("precision"), p.get("emax")], Ieee754.FORMATS.get(p.get("format") as string))));
  // A domain's name, its dialect's `Defines` entry's; null until a dialect defines it.
  assert(same([...domains.values()].map((domain) => FD.nameOf(domain)), [...domains.keys()]) && FD.nameOf(coded) === "coded");
  const loose = store.transaction(() => {
    const loose = store.Expressions.Framework.Domain((d: any) => d.kind((k: any) => k.as_value((a: any) => a.as_union(
      (u: any) => u.unicode((v: any) => v.unicode((e: any) => e))))));
    assert(FD.nameOf(loose) === null);
    store.Expressions.Unicode.Expressions.Dialect(singletons.get("Unicode"), (b: any) => b.domains({ loose: (e: any) => e.domain(loose) }));
    return loose;
  });
  assert(FD.nameOf(loose) === "loose" && FD.properties(loose).size === 0); // no property set
  raises(ValueError, () => FD.kindOf(empty), "a domain needs a kind");
  raises(ValueError, () => FD.properties(empty), "a domain needs a kind");
  // A dialect's operations by name, in order, each its expression schema, and each one's arity: Patterns', which its
  // signatures for inference name in the same order, each of the same arity.
  const patterns = store.transaction(Patterns.define)[0].get("dialect"); // again: nothing
  const operations = FD.operations(patterns);
  assert(same([...operations.keys()], [...D.SIGNATURES.keys()]) && operations.get("add") === store.registered("Expressions.Patterns.Expressions.add"));
  assert(equal(new Map([...operations].map(([name, operation]) => [name, FD.arity(operation)])),
    new Map([...D.SIGNATURES].map(([name, signature]) => [name, signature.arity()]))));
  // Patterns owns no domain, and holds no kind and no domain's schema; its family is its own, not the framework's: its
  // schemas, singleton and package are at `Expressions.Patterns.Expressions`, and nothing at the framework's old path.
  assert([...patterns.domains].length === 0 && DomainKind.branches[0].schema.branches.length > 0
    && !DomainKind.branches.map((b: any) => b.name).includes("patterns"));
  assert(await import("@mbse/expressions/Patterns/Expressions") === Patterns && patterns === store.singleton("Expressions.Patterns.Expressions"));
  for (const name of ["Expressions.Patterns.Expressions.octet", "Expressions.Patterns.Expressions.OfInteger", "Expressions.Patterns.Expressions.Packs",
    "Expressions.Framework.Patterns.Term", "Expressions.Framework.Patterns.Dialect"]) {
    raises(LookupError, () => store.registered(name), `no schema registered as '${name}'`);
  }
}

<a id="DOM-18"></a>
## DOM-18 · A language's own kinds are its dialect's: SystemVerilog's logic, Python3's, TypeScript5's, MATLAB's and Excel's strings, the tokens and Excel's errors

In [ ]:
{
  // Each dialect with kinds of its own adds its union of them to the framework's, as the branch named by its tag: a
  // language's own types (Decided, B and H), each kind's parameters, then its inferred properties, all optional.
  const languages: any = new Builders.Store();
  for (const install of [...INSTALLS, Python3.install, Matlab.install, Excel.install, Latex.install, CDialect.install, Cpp.install,
    SystemVerilog.install, TypeScript5.install]) {
    languages.transaction(install);
  }
  const LANGUAGES = new Map<string, [string, Map<string, [string, string[]]>]>([ // each dialect's tag, its path, and its kinds' properties, by tag
    ["python3", ["Expressions.Python.Python3", new Map([["str", ["OfStr", ["ordered"]]], ["nonetype", ["OfNoneType", ["ordered"]]],
      ["ellipsis", ["OfEllipsis", ["ordered"]]]])]],
    ["matlab", ["Expressions.MATLAB.Expressions", new Map([["string", ["OfString", ["ordered"]]]])]],
    ["excel", ["Expressions.Excel.Expressions", new Map([["text", ["OfText", ["ordered"]]], ["error", ["OfError", ["members", "ordered"]]]])]],
    ["c", ["Expressions.C.Expressions", new Map([["nullptr_t", ["OfNullptr", ["ordered"]]]])]],
    ["cpp", ["Expressions.Cpp.Expressions", new Map([["nullptr_t", ["OfNullptr", ["ordered"]]]])]],
    ["systemverilog", ["Expressions.SystemVerilog.Expressions", new Map([
      ["logic", ["OfLogic", ["width", "signed", "least", "greatest", "ordered"]]]])]],
    ["typescript5", ["Expressions.TypeScript.TypeScript5", new Map([["string", ["OfString", ["ordered"]]], ["null", ["OfNull", ["ordered"]]]])]],
  ]);
  const Kinds = languages.registered("Expressions.Framework.DomainKind");
  assert(same(Kinds.branches.map((b: any) => b.name), ["isoiec11404", "ieee754", "ieee1164", "unicode", ...LANGUAGES.keys()])); // LaTeX: none
  for (const [tag, [path, kinds]] of LANGUAGES) {
    const union = new Map<string, any>(Kinds.branches.map((b: any) => [b.name, b.schema])).get(tag);
    assert(union === languages.registered(`${path}.DomainKind`));
    assert(equal(new Map(union.branches.map((b: any) => [b.name, [b.schema.name, [...b.schema.properties.keys()]]])),
      new Map([...kinds].map(([kind, [schema, names]]) => [kind, [`${path}.${schema}`, names]]))));
  }
  const members = languages.registered("Expressions.Excel.Expressions.OfError").properties.get("members").schema;
  assert(members.index === languages.registered("Schemas.OfInt") && members.element === languages.registered("Schemas.OfStr"));
  // C's nullptr_t and C++'s are each their own dialect's (API, 35).
  assert(languages.registered("Expressions.C.Expressions.OfNullptr") !== languages.registered("Expressions.Cpp.Expressions.OfNullptr"));
  // A domain of each, written by hand in a transaction and owned by its dialect by name, valid against its schema;
  // `ordered` true of the strings and SystemVerilog's logic, false of the tokens and Excel's errors.
  const ERRORS = ["#DIV/0!", "#N/A", "#NAME?", "#NULL!", "#NUM!", "#REF!", "#VALUE!"];
  const WRITTEN: [[string, string], [string, (e: any) => any, Map<string, unknown>]][] = [ // by its dialect's tag and its name: its kind, its properties written, and as read
    [["systemverilog", "logic8"], ["logic", (e: any) => e.width(8n).signed(false).least(0n).greatest(255n).ordered(true),
      new Map<string, unknown>([["width", 8n], ["signed", false], ["least", 0n], ["greatest", 255n], ["ordered", true]])]],
    [["python3", "str"], ["str", (e: any) => e.ordered(true), new Map([["ordered", true]])]],
    [["python3", "NoneType"], ["nonetype", (e: any) => e.ordered(false), new Map([["ordered", false]])]],
    [["python3", "ellipsis"], ["ellipsis", (e: any) => e.ordered(false), new Map([["ordered", false]])]],
    [["typescript5", "string"], ["string", (e: any) => e.ordered(true), new Map([["ordered", true]])]],
    [["typescript5", "null"], ["null", (e: any) => e.ordered(false), new Map([["ordered", false]])]],
    [["matlab", "string"], ["string", (e: any) => e.ordered(true), new Map([["ordered", true]])]],
    [["excel", "text"], ["text", (e: any) => e.ordered(true), new Map([["ordered", true]])]],
    [["c", "nullptr_t"], ["nullptr_t", (e: any) => e.ordered(false), new Map([["ordered", false]])]],
    [["cpp", "nullptr_t"], ["nullptr_t", (e: any) => e.ordered(false), new Map([["ordered", false]])]],
    [["excel", "error"], ["error", (e: any) => e.members(ERRORS).ordered(false), new Map<string, unknown>([["members", ERRORS], ["ordered", false]])]],
  ];

  /** The builder of a schema registered under a dotted path, by its parts. */
  const schemaAt = (path: string): any => path.split(".").reduce((found: any, part) => found[part], languages);

  const LanguageDomain = languages.registered("Expressions.Framework.Domain");
  const written: any[] = languages.transaction(() => {
    const domains = WRITTEN.map(([[tag], [kind, spec]]) => languages.Expressions.Framework.Domain((d: any) => d.kind(
      (k: any) => k.as_value((a: any) => a.as_union((u: any) => u[tag]((v: any) => v[kind](spec)))))));
    WRITTEN.forEach(([[tag, name]], i) => { // each owned: written with its owner
      const path = (LANGUAGES.get(tag) as [string, unknown])[0];
      schemaAt(`${path}.Dialect`)(languages.singleton(path), (b: any) => b.domains({ [name]: (e: any) => e.domain(domains[i]) }));
    });
    return domains;
  });
  WRITTEN.forEach(([[tag, name], [kind, , read]], i) => {
    const domain = written[i];
    assert(Validators.Validate(languages)(LanguageDomain, domain).length === 0, `${tag} ${name}`);
    const [dialect, found, record] = FD.kindOf(domain); // the dialect's tag first: C's nullptr_t is not C++'s
    const [path, kinds] = LANGUAGES.get(tag) as [string, Map<string, [string, string[]]>];
    assert(dialect === tag && found === kind && record.as_schema() === languages.registered(`${path}.${(kinds.get(kind) as [string, string[]])[0]}`));
    assert(equal(FD.properties(domain), read) && FD.nameOf(domain) === name);
  });
  assert(equal(["c", "cpp"].map((tag) => FD.kindOf(written[WRITTEN.findIndex(([[t, name]]) => t === tag && name === "nullptr_t")]).slice(0, 2)),
    [["c", "nullptr_t"], ["cpp", "nullptr_t"]]));
}

<a id="DOM-21"></a>
## DOM-21 · Python3's, TypeScript5's, MATLAB's and Excel's domains are stored: each define writes its dialect's, by name, with their kinds, properties and schemas

In [ ]:
{
  // Python3's, TypeScript5's, MATLAB's and Excel's defines, each run after the installs in a transaction of its own:
  // each writes its domains, owned by its singleton by name, in order, each named as its language names its type, of a
  // standard dialect's kind or its own, with its parameters and its inferred properties; each but a token's is related
  // to its schema, an OfAny of the scalar it narrows, both ways.
  const STORED_INSTALLS = [...INSTALLS, Python3.install, Matlab.install, Excel.install, TypeScript5.install];
  const stored: any = new Builders.Store();
  for (const install of STORED_INSTALLS) stored.transaction(install);
  const [ORDERED, UNORDERED] = [new Map([["ordered", true]]), new Map([["ordered", false]])];
  const UNBOUNDED = new Map<string, unknown>([["signed", true], ["overflow", "raise"], ["ordered", true]]); // 11404's integer: no width, no least or greatest
  const BINARY64 = new Map<string, unknown>([["format", "binary64"], ["rounding", "roundTiesToEven"], ["radix", 2n],
    ["precision", 53n], ["emax", 1023n], ["ordered", true]]);
  const EXCEL_ERRORS = ["#NULL!", "#DIV/0!", "#VALUE!", "#REF!", "#NAME?", "#NUM!", "#N/A", "#GETTING_DATA", // ERROR.TYPE's
    "#SPILL!", "#CALC!", "#FIELD!", "#BLOCKED!", "#CONNECT!", "#BUSY!", "#UNKNOWN!"]; // then added since
  // Populated by hand, a standard kind's domain is as its standard dialect's own: binary64, integer, boolean, octetstring.
  assert(equal([BINARY64, UNBOUNDED, UNORDERED, ORDERED],
    ["binary64", "integer", "boolean", "octetstring"].map((name) => FD.properties(domains.get(name)))));
  type Expected = [[string, string], Map<string, unknown>, string | null];
  const STORED = new Map<any, [string, Map<string, Expected>]>([ // each dialect's path, and its domains in order: its kind, its properties, the scalar its schema narrows
    [Python3, ["Expressions.Python.Python3", new Map<string, Expected>([
      ["bool", [["isoiec11404", "boolean"], UNORDERED, "bool"]], ["int", [["isoiec11404", "integer"], UNBOUNDED, "int"]],
      ["float", [["ieee754", "ieee754"], BINARY64, "float"]], ["str", [["python3", "str"], ORDERED, "str"]],
      ["bytes", [["isoiec11404", "octetstring"], ORDERED, "bytes"]],
      ["NoneType", [["python3", "nonetype"], UNORDERED, null]], ["ellipsis", [["python3", "ellipsis"], UNORDERED, null]]])]],
    [TypeScript5, ["Expressions.TypeScript.TypeScript5", new Map<string, Expected>([
      ["boolean", [["isoiec11404", "boolean"], UNORDERED, "bool"]], ["number", [["ieee754", "ieee754"], BINARY64, "float"]],
      ["bigint", [["isoiec11404", "integer"], UNBOUNDED, "int"]], ["string", [["typescript5", "string"], ORDERED, "str"]],
      ["null", [["typescript5", "null"], UNORDERED, null]]])]],
    [Matlab, ["Expressions.MATLAB.Expressions", new Map<string, Expected>([
      ["double", [["ieee754", "ieee754"], BINARY64, "float"]], ["logical", [["isoiec11404", "boolean"], UNORDERED, "bool"]],
      ["string", [["matlab", "string"], ORDERED, "str"]]])]],
    [Excel, ["Expressions.Excel.Expressions", new Map<string, Expected>([
      ["number", [["ieee754", "ieee754"], BINARY64, "float"]], ["logical", [["isoiec11404", "boolean"], UNORDERED, "bool"]],
      ["text", [["excel", "text"], ORDERED, "str"]],
      ["error", [["excel", "error"], new Map<string, unknown>([["members", EXCEL_ERRORS], ["ordered", false]]), "str"]]])]], // an error value: its code
  ]);
  const scalarSchemas: Record<string, string> = { bool: "Schemas.OfBool", int: "Schemas.OfInt", float: "Schemas.OfFloat", str: "Schemas.OfStr", bytes: "Schemas.OfBytes" };
  const StoredDomain = stored.registered("Expressions.Framework.Domain");
  for (const [module, [path, expected]] of STORED) {
    assert(module.Expressions.define === module.define); // its package exports its module's
    const frame = stored.transaction(module.define)[0];
    const dialect = stored.singleton(path);
    assert(frame.get("dialect") === dialect && same([...dialect.domains].map((e: any) => `${e.name}`), [...expected.keys()]));
    for (const [name, [kind, properties, scalar]] of expected) {
      const domain = dialect.domains({ name });
      assert(frame.get(name) === domain && Validators.Validate(stored)(StoredDomain, domain).length === 0, `${path} ${name}`);
      assert(equal(FD.kindOf(domain).slice(0, 2), kind) && equal(FD.properties(domain), properties) && FD.nameOf(domain) === name);
      if (scalar === null) { // a token: no basic scalar holds its value, so no schema
        assert([...domain.schema].length === 0);
        raises(LookupError, () => stored.registered(`${path}.${name}`), `no schema registered as '${path}.${name}'`);
        continue;
      }
      const schema = stored.registered(`${path}.${name}`);
      assert(same(schema.schemas, [stored.registered(scalarSchemas[scalar] as string)]) && schema.validate().length === 0);
      assert(same([...domain.schema].map((e: any) => e.schema), [schema])); // HasDomain, both ways
      const read = [...new Builders.OfAny.Builder(schema, false, stored).domain].map((e: any) => e.domain);
      assert(equal(read.map((d: any) => [FD.nameOf(d), FD.kindOf(d).slice(0, 2), FD.properties(d)]), [[name, kind, properties]]));
    }
  }
  // The new kinds' records are their dialects' own.
  assert(FD.kindOf(stored.singleton("Expressions.MATLAB.Expressions").domains({ name: "string" }))[2].as_schema()
    === stored.registered("Expressions.MATLAB.Expressions.OfString"));
  assert(FD.kindOf(stored.singleton("Expressions.Excel.Expressions").domains({ name: "text" }))[2].as_schema()
    === stored.registered("Expressions.Excel.Expressions.OfText"));
  // Run again, each define writes nothing; the store round-trips whole into one with the same installs.
  const snapshot = JSON.ToJSON(stored).Snapshot();
  for (const module of STORED.keys()) stored.transaction(module.define);
  assert(JSON.ToJSON(stored).Snapshot() === snapshot);
  const again: any = new Builders.Store();
  for (const install of STORED_INSTALLS) again.transaction(install);
  JSON.FromJSON(again).Snapshot(snapshot);
  assert(JSON.ToJSON(again).Snapshot() === snapshot);
  for (const [path, expected] of STORED.values()) {
    const restored = new Map<string, any>([...again.singleton(path).domains].map((e: any) => [`${e.name}`, e.domain])); // keyed by name
    assert(equal(new Map([...expected.keys()].map((name) => [name, [FD.kindOf(restored.get(name)).slice(0, 2), FD.properties(restored.get(name))]])),
      new Map([...expected].map(([name, [kind, properties]]) => [name, [kind, properties]]))) && restored.size === expected.size);
  }
}